# 03. Siniestralidad y conglomerados

> Para los conglomerados trabajamos con los ocho años de origen **2015–2022** para disponer de trayectorias de desarrollo comparables.

In [2]:
# ------------------------------------------
# 1. Importación de librerías
# ------------------------------------------

from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display
import numpy as np

# ------------------------------------------
# 2. Configuración de rutas
# ------------------------------------------

# Localizar la carpeta principal del proyecto
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "funciones_reservas.py").exists()
)

# Permitir importar nuestras funciones
sys.path.insert(0, str(ROOT))

from funciones_reservas import leer_snapshot

# Definir las carpetas de trabajo
PROCESSED = ROOT / "data" / "processed"

# ------------------------------------------
# 3. Cargar las bases ajustadas
# ------------------------------------------

# Recuperar las bases previamente ajustadas a diciembre de 2025
emision_ajustada = leer_snapshot(
    PROCESSED / "emision_ajustada_2025.parquet"
)

siniestros_ajustados = leer_snapshot(
    PROCESSED / "siniestros_ajustados_2025.parquet"
)

# Mostrar las dimensiones de las bases
print(f"Emisión: {emision_ajustada.shape}")
print(f"Siniestros: {siniestros_ajustados.shape}")

Emisión: (21353, 11)
Siniestros: (2763, 14)


# 3.1. Preparación y cálculo de factores de siniestralidad

## 3.1.1. Factor de siniestralidad del mercado por año de origen

Primero queremos saber qué tan grandes son los siniestros respecto a las primas emitidas en cada año. Para ello sumamos los cuatro desarrollos disponibles de los siniestros y los agrupamos por año de origen.

$$\text{Factor de siniestralidad}_{t}=\frac{\text{Siniestros del año de origen }t}{\text{Prima emitida del año }t}$$

In [3]:
# ------------------------------------------
# 3.1.1 Siniestralidad total por año
# ------------------------------------------

# Columnas de desarrollo ajustadas por INPC
columnas = [f"{i}_monto pagado_ajustada" for i in range(4)]

# Sumar los desarrollos de cada registro
siniestros_ajustados["siniestro_total"] = (
    siniestros_ajustados[columnas].sum(axis=1, min_count=1)
)

# Agrupar los siniestros por año de origen
siniestros_anual = (
    siniestros_ajustados
    .groupby("año")["siniestro_total"]
    .sum()
).round(2)

# Sumar todas las primas emitidas de cada año
primas_anual = (
    emision_ajustada
    .groupby("año")["prima_emitida_ajustada"]
    .sum()
).round(2)

# Calcular el factor de siniestralidad del mercado
# Las primas iguales a cero no constituyen un denominador válido.
factor_anual = siniestros_anual / primas_anual.where(primas_anual > 0)

factor_anual.reset_index().rename(
    columns={0: "factor_siniestralidad_anual"}
).set_index("año")

,factor_siniestralidad_anual
año,
2015,1.728247
2016,0.553433
2017,0.336023
2018,0.376379
2019,0.382350
2020,0.638191
2021,0.519178
2022,0.672990
2023,0.442951


In [4]:
# ------------------------------------------
# Métricas de siniestralidad por año
# ------------------------------------------

print("Métricas de siniestralidad anual")
factor_anual.describe()

Métricas de siniestralidad anual


count    11.000000
mean      0.611773
std       0.395429
min       0.336023
25%       0.379365
50%       0.519178
75%       0.655591
max       1.728247
dtype: float64

In [5]:
# ------------------------------------------------------
# Evolución del factor de siniestralidad anual por año
# ------------------------------------------------------

fig = px.line(
    factor_anual,
    x=factor_anual.index,
    y=factor_anual.values,
    markers=True,
    title="Evolución del factor de siniestralidad anual",
    labels={"x": "Año", "y": "Factor de siniestralidad"},
)

fig.update_traces(
    line=dict(width=2, color="#636EFA"),
    marker=dict(size=6)
)

fig.update_layout(
    template="plotly",

    # Fondo
    paper_bgcolor="#F2F2F2",
    plot_bgcolor="#F2F2F2",

    # Tipografía
    font=dict(
        family="Arial",
        size=14,
        color="#2A4163"
    ),

    # Título
    title=dict(
        x=0.03,
        y=0.95,
        font=dict(size=20)
    ),

    # Espacios
    margin=dict(
        l=65,
        r=40,
        b=65,
        t=130
    ),

    # Interacción
    hovermode="x unified",
    showlegend=False,

    # Dimensiones
    height=550
)

# Eje X
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showline=False
)

# Eje Y
fig.update_yaxes(
    showgrid=True,
    gridcolor="white",
    gridwidth=1,
    zeroline=False,
    showline=False
)

fig.show()

**Observación.** El factor de siniestralidad del mercado presenta fluctuaciones importantes entre 2015 y 2025, destacando 2015 como el año con mayor siniestralidad relativa (1.72), mientras que en los años posteriores los factores se mantienen por debajo de 1.

Esto sugiere que la siniestralidad no ha seguido una tendencia constante. Para identificar qué tipos de embarcación presentan diferencias, calcularemos sus factores individuales.

## 3.1.2. Factor de siniestralidad por tipo de embarcación

Ahora queremos medir qué tan grandes son los siniestros de cada **tipo de embarcación**, considerando todos sus usos, respecto a su prima emitida en cada año de origen.

In [6]:
# ------------------------------------------
# 3.1.2 Siniestralidad por tipo de embarcación
# ------------------------------------------

# Agrupar los siniestros por año de origen y tipo de embarcación
siniestros_tipo = (
    siniestros_ajustados
    .groupby(["año", "tipo_embarcacion"])["siniestro_total"]
    .sum()
).round(2)

# Sumar las primas emitidas por año y tipo de embarcación
primas_anual_tipo = (
    emision_ajustada
    .groupby(["año", "tipo_embarcacion"])["prima_emitida_ajustada"]
    .sum()
).round(2)

# Calcular el factor por tipo de embarcación
# Si hay prima positiva y no hay siniestros registrados, el monto es cero.
# Con prima nula o negativa, el factor no se calcula.
factor_anual_tipo = (
    siniestros_tipo.reindex(primas_anual_tipo.index, fill_value=0)
    / primas_anual_tipo.where(primas_anual_tipo > 0)
)

print("Factor de siniestralidad anual por tipo de embarcación")

display(
    factor_anual_tipo.rename("factor_siniestralidad_tipo")
    .reset_index()
    .set_index(["año", "tipo_embarcacion"])
)

Factor de siniestralidad anual por tipo de embarcación


factor_siniestralidad_tipo
año  tipo_embarcacion                                   
2015 abastecedor                                0.233185
     artefactos navales                              NaN
     atunero                                    0.045858
     barcaza o chalan                           0.111802
     barco-escuela                              0.000000
...                                                  ...
2025 remolcador                                 0.477299
     sardinero o anchovetero                    0.111391
     tiburoneros                                0.000000
     velero                                     0.693984
     yate                                       1.604225

[286 rows x 1 columns]

## 3.1.3. Selección de los tipos de embarcación comparables

Identificamos los que registran prima emitida positiva en **los ocho años de 2015 a 2022**. También calculamos qué proporción de la prima emitida del mercado representa esta selección.

In [7]:
# ------------------------------------------
# 3.1.3 Identificación de tipos de embarcación
# ------------------------------------------

# Convertir las primas agrupadas en un DataFrame
emision_tipos = primas_anual_tipo.reset_index()

# Identificar los tipos de embarcación en la base
tipos = emision_tipos[["tipo_embarcacion"]].drop_duplicates()
print("Número de tipos de embarcación:", len(tipos))

# --------------------------------------------------
# Cobertura por tipo durante 2015-2022
# --------------------------------------------------

# Seleccionar los años con desarrollo completo
periodo = emision_tipos[
    emision_tipos["año"].between(2015, 2022)
].copy()

# Contar años con prima positiva por tipo
cobertura_8 = (
    periodo
    .assign(positiva=periodo["prima_emitida_ajustada"] > 0)
    .groupby("tipo_embarcacion")["positiva"]
    .sum()
)

# Identificar tipos con prima positiva en los ocho años
tipos_completos = cobertura_8[cobertura_8 == 8].index

print("Tipos completos 2015-2022:", len(tipos_completos))

# --------------------------------------------------
# Participación de los tipos seleccionados
# --------------------------------------------------

seleccion = periodo["tipo_embarcacion"].isin(tipos_completos)

prima_mercado = periodo["prima_emitida_ajustada"].sum()
prima_tipos = periodo.loc[
    seleccion, "prima_emitida_ajustada"
].sum()

participacion = (
    prima_tipos / prima_mercado * 100
    if prima_mercado != 0 else np.nan
)

print(f"Participación de los tipos completos: {participacion:.2f}%")

Número de tipos de embarcación: 32
Tipos completos 2015-2022: 19
Participación de los tipos completos: 96.29%


**Conclusión.** Los tipos de embarcación seleccionados presentan primas emitidas positivas durante los ocho años de estudio y representan el 96.29% de las primas del periodo. Por ello, se considera que la selección ofrece una cobertura económica suficiente para continuar con el análisis de conglomerados, conservando trayectorias temporales comparables.

In [8]:
# --------------------------------------------------
# 3.1.3. Tipos excluidos y motivo de su exclusión
# --------------------------------------------------

# Una fila por tipo y una columna por año de origen
primas_matriz = (
    primas_anual_tipo
    .unstack("año")
    .reindex(columns=range(2015, 2023))
)

# Resumir la situación de cada tipo en los ocho años
detalle_tipos = pd.DataFrame({
    "Años positivos": primas_matriz.gt(0).sum(axis=1),
    "Años en cero": primas_matriz.eq(0).sum(axis=1),
    "Años negativos": primas_matriz.lt(0).sum(axis=1),
    "Años sin registro": primas_matriz.isna().sum(axis=1)
})

# Identificar los años exactos que impiden tener trayectoria completa
anios = list(range(2015, 2023))
detalle_tipos["Años sin prima positiva"] = primas_matriz.apply(
    lambda fila: ", ".join(
        str(anio) for anio in anios
        if pd.isna(fila[anio]) or fila[anio] <= 0
    ),
    axis=1
)

# Describir la causa observada, sin asumir que es un error
# (un tipo puede presentar más de una situación)
def explicar_exclusion(fila):
    motivos = []
    if fila["Años sin registro"] > 0:
        motivos.append("Años sin registro de emisión")
    if fila["Años en cero"] > 0:
        motivos.append("Prima anual igual a cero")
    if fila["Años negativos"] > 0:
        motivos.append("Prima anual negativa")
    return "; ".join(motivos)

detalle_tipos["Motivo"] = detalle_tipos.apply(
    explicar_exclusion, axis=1
)

# Mostrar los tipos que no cumplen el criterio de ocho años positivos
tipos_excluidos = detalle_tipos.loc[
    ~detalle_tipos.index.isin(tipos_completos)
].copy()

print("Tipos seleccionados:", len(tipos_completos))
print("Tipos excluidos de la matriz:", len(tipos_excluidos))

display(
    tipos_excluidos.reset_index()
    .sort_values(["Años positivos", "tipo_embarcacion"])
)

# Comprobación: los tipos seleccionados y excluidos cubren
# todos los tipos que aparecen en la base de emisión durante 2015-2022.
assert len(tipos_completos) + len(tipos_excluidos) == len(primas_matriz)


Tipos seleccionados: 19
Tipos excluidos de la matriz: 13


,tipo_embarcacion,Años positivos,Años en cero,Años negativos,Años sin registro,Años sin prima positiva,Motivo
3,buque frigorifico,0,0,1,7,"2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022",Años sin registro de emisión; Prima anual nega...
11,porta contenedor,1,0,0,7,"2015, 2016, 2017, 2018, 2020, 2021, 2022",Años sin registro de emisión
5,buque-tanque quimico,2,0,0,6,"2015, 2018, 2019, 2020, 2021, 2022",Años sin registro de emisión
7,carguero tipo roll on - roll off,2,0,0,6,"2015, 2016, 2017, 2018, 2019, 2020",Años sin registro de emisión
2,barco-hidrografico,3,0,0,5,"2015, 2017, 2020, 2021, 2022",Años sin registro de emisión
6,buque-tanque-minerales,3,0,0,5,"2015, 2016, 2017, 2021, 2022",Años sin registro de emisión
4,buque transportador de autos,4,0,2,2,"2016, 2019, 2021, 2022",Años sin registro de emisión; Prima anual nega...
12,tiburoneros,4,0,0,4,"2016, 2017, 2018, 2019",Años sin registro de emisión
0,artefactos navales,6,0,1,1,"2015, 2019",Años sin registro de emisión; Prima anual nega...
1,barco-escuela,6,0,1,1,"2016, 2022",Años sin registro de emisión; Prima anual nega...


## 3.1.4. Construcción de la matriz para el análisis multivariado

Cada fila será un **tipo de embarcación** seleccionado y cada columna representará su factor de siniestralidad en un año de 2015 a 2022. La matriz tendrá **ocho columnas** y tantas filas como tipos completos se hayan identificado.

In [9]:
# ---------------------------------------------------------------------
# DataFrame del factor de siniestralidad de los tipos seleccionados
# ---------------------------------------------------------------------

# Convertir los factores en DataFrame
df_factores = factor_anual_tipo.rename("factor_siniestralidad").reset_index()

# Identificar los tipos seleccionados
filtro = df_factores["tipo_embarcacion"].isin(tipos_completos)

# Filtrar los tipos y años de interés
df_tipos = df_factores.loc[
    filtro & df_factores["año"].between(2015, 2022)
].copy()

# Si hay prima positiva y no hay siniestros registrados, el factor es 0.
df_tipos["factor_siniestralidad"] = (
    df_tipos["factor_siniestralidad"].fillna(0)
)

In [10]:
# ---------------------------------------------------------------
# Mover el factor de cada año a una columna separada
# ---------------------------------------------------------------

df_tipos_pivot = df_tipos.pivot(
    index="tipo_embarcacion",
    columns="año",
    values="factor_siniestralidad"
)

df_tipos_pivot = df_tipos_pivot.reindex(
    columns=range(2015, 2023)
)

df_tipos_pivot.rename(
    columns={i: f"fs_{i}" for i in range(2015, 2023)},
    inplace=True
)

print("Dimensiones de la matriz:", df_tipos_pivot.shape)
display(df_tipos_pivot)

Dimensiones de la matriz: (19, 8)


año,fs_2015,fs_2016,fs_2017,fs_2018,fs_2019,fs_2020,fs_2021,fs_2022
tipo_embarcacion,,,,,,,,
abastecedor,0.233185,0.377512,0.438483,0.348956,0.034605,0.560429,0.571040,0.551124
atunero,0.045858,0.267407,0.080078,0.371242,0.264100,2.607975,4.002534,3.446874
barcaza o chalan,0.111802,2.137235,0.000000,0.000000,0.380713,0.000000,2.237615,0.052041
buque-tanque,35.767174,1.057684,0.708277,3.470587,6.861467,1.761038,0.163708,0.065368
buque-tanque petrolero y productos derivados,0.000735,0.122723,0.360971,0.182118,0.655586,0.033952,2.038834,0.038514
camaronero,0.426566,1.046860,0.800984,0.041806,0.241447,0.488655,0.114397,0.144984
carguero a granel,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
carguero general,1.464266,0.197423,0.062055,0.082406,0.941033,0.447100,0.004004,0.145652
dragas,0.000000,12.235499,0.000000,5.259638,1.178765,0.568701,0.000000,0.000000


# 3.2 Preprocesamiento Multivariado

## 3.2.1. Formulación del problema

El objetivo es identificar grupos de tipos de embarcación con comportamientos semejantes de siniestralidad, buscando formar conjuntos de riesgos homogéneos que puedan apoyar la estimación de reservas de riesgos en curso.

Para ello, utilizaremos los factores de siniestralidad de 2015 a 2022, donde cada tipo de embarcación representa una observación y cada año una variable.

Antes de aplicar los métodos de agrupación, analizaremos las relaciones entre las variables y la presencia de valores extremos, para determinar cómo preparar adecuadamente los datos.

## 3.2.2 Construcción de caracteristicas

A partir de los factores de siniestralidad, construiremos 4 características que resuman el comportamiento de la siniestralidad de cada tipo de embarcación: 

- **Siniestralidad acumulada**: mide qué proporción de las primas emitidas representan los siniestros durante los 8 años
- **Variabilidad temporal**: mide cuánto fluctúa el factor de siniestralidad entre años
- **recurrencia de alta siniestralidad**: mide la proporción de años en los que los factores de siniestralidad superan 1
- **Tendencia**: mide si el factor de siniestralidad tiende a aumentar o disminuir durante el periodo
- **Prima acumulada**: volumen total de primas emitidas durante un periodo, esta ayuda para evaluar cuanta prima acumulada cada tipo y cada conglomerado para identificar cuales tienen poca experiencia economica. 

In [23]:
# --------------------------------------------------
# 3.2.2. Construcción de características
# --------------------------------------------------

anios = list(range(2015, 2023))

# Matriz de factores anuales
X = df_tipos_pivot.copy()

# Recuperar primas y siniestros por tipo y año
primas_matriz = (
    primas_anual_tipo
    .unstack("año")
    .reindex(index=X.index, columns=anios)
)

siniestros_matriz = (
    siniestros_tipo
    .unstack("año")
    .reindex(index=X.index, columns=anios)
    .fillna(0)
)

# --------------------------------------------------
# Construcción de características
# --------------------------------------------------

caracteristicas = pd.DataFrame(index=X.index)

# 1. Siniestralidad acumulada
caracteristicas["Siniestralidad acumulada"] = (
    siniestros_matriz.sum(axis=1)
    / primas_matriz.sum(axis=1)
)

# 2. Variabilidad temporal
caracteristicas["Variabilidad temporal"] = (
    X.std(axis=1, ddof=1)
)

# 3. Recurrencia de alta siniestralidad
caracteristicas["Recurrencia"] = (
    (X > 1).mean(axis=1)
)

# 4. Tendencia temporal (pendiente anual)
caracteristicas["Tendencia"] = X.apply(
    lambda fila: np.polyfit(
        range(8), fila.to_numpy(dtype=float), 1
    )[0],
    axis=1
)

# --------------------------------------------------
# Indicador auxiliar de experiencia
# --------------------------------------------------

experiencia = pd.DataFrame(index=X.index)

experiencia["Prima acumulada"] = (
    primas_matriz.sum(axis=1)
)

# --------------------------------------------------
# Resultados
# --------------------------------------------------

print("Matriz de características:")
display(caracteristicas.round(4))

print("Indicador auxiliar de experiencia:")
display(experiencia.round(2))

caracteristicas_experiencia = pd.concat(
    [caracteristicas, experiencia], axis=1
)

Matriz de características:


,Siniestralidad acumulada,Variabilidad temporal,Recurrencia,Tendencia
tipo_embarcacion,,,,
abastecedor,0.2774,0.1863,0.000,0.0386
atunero,0.8627,1.6745,0.375,0.5948
barcaza o chalan,0.2143,0.9791,0.250,0.0055
buque-tanque,7.0308,12.1448,0.625,-2.9504
buque-tanque petrolero y productos derivados,0.2045,0.6860,0.125,0.1112
camaronero,0.3754,0.3559,0.125,-0.0877
carguero a granel,0.0000,0.0000,0.000,0.0000
carguero general,0.2538,0.5213,0.125,-0.0974
dragas,1.6985,4.3556,0.375,-0.7566


Indicador auxiliar de experiencia:


,Prima acumulada
tipo_embarcacion,
abastecedor,7.288401e+09
atunero,4.672413e+08
barcaza o chalan,2.008418e+08
buque-tanque,6.430191e+08
buque-tanque petrolero y productos derivados,3.249467e+09
camaronero,1.822178e+08
carguero a granel,1.587132e+07
carguero general,5.463084e+08
dragas,3.400357e+07


## 3.2.2 Diagnóstico Multivariado

In [28]:
# --------------------------------------------------
# Distribuciones de las características
# --------------------------------------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Colores fijos para cada característica
colores_caracteristicas = {
    "Siniestralidad acumulada": "#636EFA",
    "Variabilidad temporal": "#00BFA6",
    "Recurrencia": "#EF9B38",
    "Tendencia": "#AB63FA"
}

# Crear figura con cuatro filas independientes
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08
)

# Crear un boxplot horizontal por característica
for i, (columna, color) in enumerate(
    colores_caracteristicas.items(), start=1
):
    fig.add_trace(
        go.Box(
            x=caracteristicas[columna],
            orientation="h",
            name=columna,
            boxpoints="all",
            jitter=0.4,
            pointpos=-1.3,
            line=dict(color=color, width=2),
            marker=dict(color=color, size=7, opacity=0.6),
            fillcolor=color,
            opacity=0.65,
            showlegend=True,
            # Identificar cada tipo de embarcación
        customdata=caracteristicas.index.to_numpy().reshape(-1, 1),
        hoveron="points",
        hovertemplate="<b>Tipo:</b> %{customdata[0]}<br><b>Valor:</b> %{x:.4f}<extra></extra>"

        ),
        row=i,
        col=1
    )

# Diseño general
fig.update_layout(
    template="plotly",
    title=dict(
        text="Distribución de las características de siniestralidad",
        x=0.03,
        y=0.98,
        font=dict(size=20)
    ),
    paper_bgcolor="#F2F2F2",
    plot_bgcolor="#F2F2F2",
    font=dict(
        family="Arial",
        size=14,
        color="#2A4163"
    ),
    margin=dict(l=75, r=50, b=60, t=125),
    height=650,
    width=1000
)


# Escala y cuadrícula independiente para cada característica
fig.update_xaxes(
    matches=None,
    showgrid=True,
    gridcolor="white",
    gridwidth=1,
    zeroline=True,
    showline=True
)

fig.update_yaxes(
    showgrid=False,
    showticklabels=False,
    zeroline=True,
    title=None
)


fig.show()

In [29]:
# --------------------------------------------------
# Matrices de correlación: Pearson y Spearman
# --------------------------------------------------

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Calcular correlaciones
correlacion_pearson = caracteristicas_experiencia.corr(method="pearson")
correlacion_spearman = caracteristicas_experiencia.corr(method="spearman")

# Crear figura con dos matrices
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Pearson", "Spearman"],
    horizontal_spacing=0.15
)

# Agregar matrices
for i, matriz in enumerate(
    [correlacion_pearson, correlacion_spearman], start=1
):
    fig.add_trace(
        go.Heatmap(
            z=matriz.values,
            x=matriz.columns,
            y=matriz.index,
            text=matriz.values,
            texttemplate="%{text:.2f}",
            textfont=dict(size=14),
            coloraxis="coloraxis",
            xgap=3,
            ygap=3,
            hovertemplate=(
                "%{y} / %{x}<br>"
                "Correlación: %{z:.3f}"
                "<extra></extra>"
            )
        ),
        row=1,
        col=i
    )

# Diseño general
fig.update_layout(
    template="plotly",

    # Fondo
    paper_bgcolor="#F2F2F2",
    plot_bgcolor="#F2F2F2",

    # Tipografía
    font=dict(
        family="Arial",
        size=14,
        color="#2A4163"
    ),

    # Título
    title=dict(
        text="Correlaciones entre características de siniestralidad",
        x=0.03,
        y=0.97,
        font=dict(size=20)
    ),

    # Escala de color compartida
    coloraxis=dict(
        colorscale="PuOr",
        cmin=-1,
        cmax=1,
        cmid=0,
        colorbar=dict(
            title="Correlación",
            thickness=15,
            len=0.75
        )
    ),

    # Dimensiones y espacios
    width=1450,
    height=650,
    margin=dict(l=180, r=100, b=140, t=130),

    showlegend=False
)

# Ejes
fig.update_xaxes(
    tickangle=-35,
    showgrid=False,
    zeroline=False
)

fig.update_yaxes(
    autorange="reversed",
    showgrid=False,
    zeroline=False
)

# Subtítulos
fig.update_annotations(
    font=dict(size=17, color="#2A4163")
)

fig.show()

Las distribuciones muestran valores extremos, principalmente en siniestralidad acumulada, variabilidad temporal y tendencia. Además, existe una correlación elevada entre variabilidad y recurrencia (Pearson = 0.84, Spearman = 0.92). Por otro lado, la relación entre siniestralidad acumulada y tendencia cambia considerablemente entre Pearson (−0.91) y Spearman (−0.08).

Estas diferencias sugieren que algunos valores extremos podrían estar influyendo en las correlaciones de Pearson. Asimismo, la fuerte relación entre variabilidad y recurrencia indica que ambas características podrían estar aportando información similar.

Antes de construir los conglomerados, es necesario evaluar la influencia de los valores extremos y la posible redundancia entre características, para evitar que determinadas observaciones o aspectos del riesgo dominen las agrupaciones.

## 3.2.3 Preprocesamiento: Tranformación logaritmica y estandarización

Las distribuciones presentan asimetría y valores extremos, principalmente en siniestralidad acumulada y variabilidad temporal. Por ello, aplicamos una transformación logarítmica selectiva para reducir la influencia de las observaciones de mayor magnitud, conservando la recurrencia y la tendencia en sus escalas originales.

Posteriormente, estandarizamos las características para hacerlas comparables y evitar que las diferencias de escala dominen las distancias. Se conservarán los escenarios original y transformado para evaluar la sensibilidad de los conglomerados.

In [14]:
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------
# 3.2.4. Transformación y estandarización
# --------------------------------------------------

# Escenario A: características originales
X_original = caracteristicas.copy()

# Escenario B: transformación logarítmica selectiva
X_log = caracteristicas.copy()

columnas_log = [
    "Siniestralidad acumulada",
    "Variabilidad temporal"
]

X_log[columnas_log] = np.log1p(X_log[columnas_log])

# --------------------------------------------------
# Estandarización de ambos escenarios
# --------------------------------------------------

X_original_std = pd.DataFrame(
    StandardScaler().fit_transform(X_original),
    index=X_original.index,
    columns=X_original.columns
)

X_log_std = pd.DataFrame(
    StandardScaler().fit_transform(X_log),
    index=X_log.index,
    columns=X_log.columns
)

# --------------------------------------------------
# Resultados
# --------------------------------------------------

print("Características transformadas y estandarizadas:")
display(X_log_std.round(4))

Características transformadas y estandarizadas:


,Siniestralidad acumulada,Variabilidad temporal,Recurrencia,Tendencia
tipo_embarcacion,,,,
abastecedor,-0.5292,-0.7857,-1.1371,0.2430
atunero,0.3434,0.3751,1.0234,1.0076
barcaza o chalan,-0.6465,-0.0549,0.3032,0.1975
buque-tanque,3.7237,2.6488,2.4638,-3.8664
buque-tanque petrolero y productos derivados,-0.6652,-0.2837,-0.4170,0.3428
camaronero,-0.3581,-0.5949,-0.4170,0.0693
carguero a granel,-1.0955,-1.0296,-1.1371,0.1899
carguero general,-0.5724,-0.4305,-0.4170,0.0560
dragas,1.2008,1.3667,1.0234,-0.8502


In [15]:
# --------------------------------------------------
# Distribuciones de las características
# --------------------------------------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Colores fijos para cada característica
colores_caracteristicas = {
    "Siniestralidad acumulada": "#636EFA",
    "Variabilidad temporal": "#00BFA6",
    "Recurrencia": "#EF9B38",
    "Tendencia": "#AB63FA"
}

# Crear figura con cuatro filas independientes
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08
)

# Crear un boxplot horizontal por característica
for i, (columna, color) in enumerate(
    colores_caracteristicas.items(), start=1
):
    fig.add_trace(
        go.Box(
            x=X_log_std[columna],
            orientation="h",
            name=columna,
            boxpoints="all",
            jitter=0.4,
            pointpos=-1.3,
            line=dict(color=color, width=2),
            marker=dict(color=color, size=7, opacity=0.6),
            fillcolor=color,
            opacity=0.65,
            showlegend=True,
            # Identificar cada tipo de embarcación
        customdata=caracteristicas.index.to_numpy().reshape(-1, 1),
        hoveron="points",
        hovertemplate="<b>Tipo:</b> %{customdata[0]}<br><b>Valor:</b> %{x:.4f}<extra></extra>"

        ),
        row=i,
        col=1
    )

# Diseño general
fig.update_layout(
    template="plotly",
    title=dict(
        text="Distribución de las características de siniestralidad",
        x=0.03,
        y=0.98,
        font=dict(size=20)
    ),
    paper_bgcolor="#F2F2F2",
    plot_bgcolor="#F2F2F2",
    font=dict(
        family="Arial",
        size=14,
        color="#2A4163"
    ),
    margin=dict(l=75, r=50, b=60, t=125),
    height=650,
    width=1000
)


# Escala y cuadrícula independiente para cada característica
fig.update_xaxes(
    matches=None,
    showgrid=True,
    gridcolor="white",
    gridwidth=1,
    zeroline=True,
    showline=True
)

fig.update_yaxes(
    showgrid=False,
    showticklabels=False,
    zeroline=True,
    title=None
)


fig.show()

# 3.3 Conglomerados

Se analizarán 2 escenarios de construccion de conglomerados con el fin de analizar la sensibilidad respecto a los valores extremos: 

- Escenario A: Tranformada con extremos
- Escenario B: Transformada sin extremos 

En cada escenario se aplican distintos metodos de agrupación (Ward, Liga Completa, K-means, K-medoids) evaluados a distintos valores de k (2-10) con las métricas Silhouette, Calinski-Harabasz, WCSS, y tamaños de los grupos 

- **Silhouette Score**: Compara si cada observación esta más cerca de los integrantes de su propio grupo que de los integrantes de otros grupos 

$$s(i) = \frac{b(i)-a(i)}{max\{b(i), a(i)\}}, \qquad s(i) \in [0,1]$$

Donde: $s(i)$ es el score, $b(i)$ es la distacia promedio del punto i al grupo vecino más cercano, $a(i)$ es la distancia promedio del punto i a los puntos de su mismo cluster

- **Calinski-Harabasz**: Evalua si la variabilidad entre los grupos es grande en comparación con la variabilidad dentro de ellos

$$CH = \frac{B/(k-1)}{W/(n-k)}$$

Donde: $B$ es la suma de cuadrados entre grupos, $W$ la suma de cuadrados dentro de los grupos (WCSS), $n$ número de observaciones y $k$ número de conglomerados 

- **WCSS**: Evalua que tan dispersas estan las observaciones respecto al centro de sus propios conglomerados:

$$WCSS = \sum^{k}_{g=1} \sum_{i \in C_g} (x_i - \bar{x}_g)^2$$

In [19]:
# --------------------------------------------------
# 3.3 y 3.4. Comparación de escenarios
# --------------------------------------------------

from sklearn.preprocessing import StandardScaler
from IPython.display import display, HTML
from funciones_reservas import evaluar_modelo

# --------------------------------------------------
# 1. Identificar observaciones extremas
# --------------------------------------------------

# Estandarizar características originales para detectar extremos
X_original_std = pd.DataFrame(
    StandardScaler().fit_transform(caracteristicas),
    index=caracteristicas.index,
    columns=caracteristicas.columns
)

# Criterio exploratorio: |z| > 3 en alguna característica
tipos_extremos = X_original_std.index[
    (X_original_std.abs() > 3).any(axis=1)
]

print("Tipos excluidos temporalmente en el escenario B:")
display(X_original_std.loc[tipos_extremos].round(3))

# --------------------------------------------------
# 2. Construir los dos escenarios
# --------------------------------------------------

# A: transformación y estandarización, con todos los tipos
X_A = X_log_std.copy()

# B: sin transformación y sin extremos, reestandarizado
X_B_original = caracteristicas.drop(index=tipos_extremos)

X_B = pd.DataFrame(
    StandardScaler().fit_transform(X_B_original),
    index=X_B_original.index,
    columns=X_B_original.columns
)

escenarios = {
    "A": X_A,
    "B": X_B
}

print(f"Escenario A: {len(X_A)} tipos de embarcación")
print(f"Escenario B: {len(X_B)} tipos de embarcación")

# --------------------------------------------------
# 3. Evaluar los cuatro métodos
# --------------------------------------------------

metodos = [
    "Ward",
    "Liga completa",
    "K-means",
    "K-medoids"
]

# Mismo rango de k, compatible con ambos escenarios
k_max = min(10, len(X_A) - 1, len(X_B) - 1)

resultados = {}

for metodo in metodos:

    print(f"\n====================================================================== {metodo.upper()} ======================================================================\n")

    # Evaluación de cada escenario
    tabla_A = evaluar_modelo(X_A, metodo, k_max)
    tabla_B = evaluar_modelo(X_B, metodo, k_max)

    # Guardar resultados para utilizarlos después
    resultados[(metodo, "A")] = tabla_A
    resultados[(metodo, "B")] = tabla_B

    # Mostrar tablas lado a lado
    display(HTML(f"""
    <style>
        .comparacion {{
            display: flex;
            gap: 20px;
            align-items: flex-start;
            width: 100%;
        }}
        .escenario {{
            flex: 1;
            min-width: 0;
            overflow-x: auto;
        }}
        .escenario h4 {{
            font-family: Arial;
            color: #2A4163;
            margin-bottom: 12px;
        }}
        .tabla-metrica {{
            border-collapse: collapse;
            width: 100%;
            font-family: Arial;
            font-size: 12px;
        }}
        .tabla-metrica th,
        .tabla-metrica td {{
            padding: 8px;
            border-bottom: 1px solid #ddd;
            text-align: center;
        }}
        .tabla-metrica th {{
            background-color: #E4EAF3;
            color: #2A4163;
        }}
        .tabla-metrica tr:hover {{
            background-color: #EDF1F8;
        }}
    </style>

    <div class="comparacion">

        <div class="escenario">
            <h4>Escenario A — Transformado, con extremos</h4>
            {tabla_A.to_html(index=False, classes="tabla-metrica")}
        </div>

        <div class="escenario">
            <h4>Escenario B — Original, sin extremos</h4>
            {tabla_B.to_html(index=False, classes="tabla-metrica")}
        </div>

    </div>
    """))

Tipos excluidos temporalmente en el escenario B:


,Siniestralidad acumulada,Variabilidad temporal,Recurrencia,Tendencia
tipo_embarcacion,,,,
buque-tanque,4.124,3.146,2.464,-3.866


Escenario A: 19 tipos de embarcación
Escenario B: 18 tipos de embarcación

====================================================================== WARD ======================================================================



Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
2,0.717,23.667,31.770,"[18, 1]",1
3,0.539,35.340,14.029,"[15, 3, 1]",1
4,0.374,36.912,9.067,"[10, 5, 3, 1]",1
5,0.365,42.700,5.758,"[10, 5, 2, 1, 1]",1
6,0.340,54.152,3.482,"[10, 5, 1, 1, 1, 1]",1
7,0.342,73.515,2.013,"[5, 5, 5, 1, 1, 1, 1]",1
8,0.318,77.614,1.508,"[5, 5, 4, 1, 1, 1, 1, 1]",1
9,0.307,84.454,1.108,"[5, 4, 4, 1, 1, 1, 1, 1, 1]",1
10,0.278,92.792,0.810,"[4, 4, 4, 1, 1, 1, 1, 1, 1, 1]",1
Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo



====================================================================== LIGA COMPLETA ======================================================================



Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
2,0.717,23.667,31.770,"[18, 1]",1
3,0.539,35.340,14.029,"[15, 3, 1]",1
4,0.515,30.449,10.720,"[15, 2, 1, 1]",1
5,0.370,28.002,8.444,"[15, 1, 1, 1, 1]",1
6,0.340,54.152,3.482,"[10, 5, 1, 1, 1, 1]",1
7,0.269,49.055,2.977,"[10, 4, 1, 1, 1, 1, 1]",1
8,0.318,77.614,1.508,"[5, 5, 4, 1, 1, 1, 1, 1]",1
9,0.307,84.454,1.108,"[5, 4, 4, 1, 1, 1, 1, 1, 1]",1
10,0.278,92.792,0.810,"[4, 4, 4, 1, 1, 1, 1, 1, 1, 1]",1
Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo



====================================================================== K-MEANS ======================================================================



Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
2,0.717,23.667,31.770,"[18, 1]",1
3,0.539,35.340,14.029,"[15, 3, 1]",1
4,0.370,37.661,8.908,"[10, 6, 2, 1]",1
5,0.367,45.841,5.391,"[10, 6, 1, 1, 1]",1
6,0.340,54.152,3.482,"[10, 5, 1, 1, 1, 1]",1
7,0.342,73.515,2.013,"[5, 5, 5, 1, 1, 1, 1]",1
8,0.318,77.614,1.508,"[5, 5, 4, 1, 1, 1, 1, 1]",1
9,0.307,84.454,1.108,"[5, 4, 4, 1, 1, 1, 1, 1, 1]",1
10,0.278,92.792,0.810,"[4, 4, 4, 1, 1, 1, 1, 1, 1, 1]",1
Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo



====================================================================== K-MEDOIDS ======================================================================



Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
2,0.717,23.667,31.770,"[18, 1]",1
3,0.291,24.221,18.870,"[10, 8, 1]",1
4,0.293,29.665,10.962,"[9, 8, 1, 1]",1
5,0.237,28.515,8.309,"[7, 5, 5, 1, 1]",1
6,0.315,53.196,3.542,"[6, 5, 5, 1, 1, 1]",1
7,0.342,73.515,2.013,"[5, 5, 5, 1, 1, 1, 1]",1
8,0.319,72.466,1.613,"[5, 5, 4, 1, 1, 1, 1, 1]",1
9,0.289,79.593,1.175,"[5, 4, 4, 1, 1, 1, 1, 1, 1]",1
10,0.300,91.615,0.821,"[5, 4, 3, 1, 1, 1, 1, 1, 1, 1]",1
Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo


En el escenario A, los cuatro métodos generan dos grupos de 18 y 1 tipos de embarcación, con un Silhouette de 0.717. En el escenario B aparecen agrupaciones más distribuidas, destacando Ward y K-means con dos grupos de 15 y 3 tipos y un Silhouette de 0.578. Al aumentar el número de conglomerados, ambos escenarios presentan grupos con una sola observación.

Los resultados muestran que las agrupaciones del escenario A están dominadas por una observación extrema, lo que explica su elevada separación. En el escenario B se identifican estructuras menos desequilibradas, aunque persisten diferencias importantes entre los métodos y la formación de grupos pequeños.

Los resultados sugieren que los valores extremos y las decisiones de preprocesamiento influyen considerablemente en la formación de conglomerados. Por ello, todavía no es posible seleccionar una solución definitiva únicamente mediante las métricas, siendo necesario revisar la composición y las características de los grupos para evaluar su homogeneidad e interpretación actuarial.